# LeetCode 3065 — Minimum Operations to Exceed Threshold Value I bằng Python

Link đề: https://leetcode.com/problems/minimum-operations-to-exceed-threshold-value-i/ · Độ khó: **Easy** · Chủ đề: Array, Simulation, Sorting

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def minOperations(self, nums: List[int], k: int) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi thao tác **xóa một phần tử nhỏ nhất** của mảng; tìm **số thao tác ít nhất** để **mọi phần tử còn lại đều `>= k`**.
2. Nắm các công cụ Python: `min`, `list.remove`, `sorted`, `bisect.bisect_left` (tìm vị trí trên mảng đã sắp xếp), `heapq` (hàng đợi ưu tiên), và đếm bằng `sum` trên biểu thức boolean.
3. Đi từ cách **mô phỏng đúng đề bài** (lặp: tìm min, xóa) → **sắp xếp rồi bỏ từng phần tử nhỏ** → **`bisect`** → **min-heap** → **một dòng `sum(x < k for x in nums)`**.
4. Hiểu mấu chốt: mỗi lần xóa min thì **mọi phần tử `< k` đều phải bị xóa**, và chỉ chúng thôi, nên đáp án là **số phần tử nhỏ hơn `k`**.
5. Tránh các lỗi kinh điển: đếm `<= k` thay vì `< k`, đếm phần tử `>= k`, vừa duyệt vừa xóa phần tử khỏi list, và nhầm "bằng `k`" là chưa đạt.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` (chỉ số từ `0`) và một số nguyên `k`.

Trong **một thao tác**, bạn được **xóa một phần tử nhỏ nhất** của `nums`.

Trả về **số thao tác tối thiểu** để **tất cả phần tử còn lại** của `nums` đều **lớn hơn hoặc bằng `k`**.

**Ví dụ 1**

- Input: `nums = [2, 11, 10, 1, 3]`, `k = 10`
- Output: `3`
- Giải thích: sau thao tác 1 mảng là `[2, 11, 10, 3]`; sau thao tác 2 là `[11, 10, 3]`; sau thao tác 3 là `[11, 10]`. Giờ mọi phần tử đều `>= 10`. Cần `3` thao tác.

**Ví dụ 2**

- Input: `nums = [1, 1, 2, 4, 9]`, `k = 1`
- Output: `0`
- Giải thích: mọi phần tử đã `>= 1`, không cần thao tác.

**Ví dụ 3**

- Input: `nums = [1, 1, 2, 4, 9]`, `k = 9`
- Output: `4`
- Giải thích: chỉ phần tử `9` thỏa `>= 9`; ta phải xóa `4` phần tử còn lại (`1, 1, 2, 4`).

**Ràng buộc:** `1 <= nums.length <= 50`; `1 <= nums[i] <= 10^9`; `1 <= k <= 10^9`; đề đảm bảo có **ít nhất một** phần tử `nums[i] >= k`.

In [1]:
from typing import List
import bisect
import heapq
import random

examples = [
    ([2, 11, 10, 1, 3], 10, 3),
    ([1, 1, 2, 4, 9], 1, 0),
    ([1, 1, 2, 4, 9], 9, 4),
]
for nums, k, expected in examples:
    print("nums =", nums, "k =", k, "-> kỳ vọng", expected)

nums = [2, 11, 10, 1, 3] k = 10 -> kỳ vọng 3
nums = [1, 1, 2, 4, 9] k = 1 -> kỳ vọng 0
nums = [1, 1, 2, 4, 9] k = 9 -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1. `min` và `list.remove`

- `min(a)` trả về phần tử **nhỏ nhất** của `a` (lỗi `ValueError` nếu `a` rỗng).
- `a.remove(x)` **xóa lần xuất hiện đầu tiên** của giá trị `x` khỏi `a` (thay đổi `a` tại chỗ, không trả giá trị). Nếu không có `x` thì báo `ValueError`.

Nếu có nhiều phần tử bằng nhau cùng là nhỏ nhất, `remove(min(a))` chỉ xóa **một** trong số chúng, đúng như đề ("xóa một phần tử nhỏ nhất").

In [2]:
a = [2, 11, 10, 1, 3]
m = min(a)
print("nhỏ nhất:", m)           # 1
a.remove(m)
print("sau khi xóa:", a)        # [2, 11, 10, 3]
b = [1, 1, 2]
b.remove(min(b))
print(b)                        # [1, 2]  (chỉ xóa MỘT số 1)

nhỏ nhất: 1
sau khi xóa: [2, 11, 10, 3]
[1, 2]


### 3.2. `sorted` và bỏ phần tử nhỏ nhất ở đầu

`sorted(a)` trả về **bản sao đã sắp xếp tăng dần** (không đổi `a`). Phần tử nhỏ nhất luôn ở đầu (chỉ số `0`). Xóa "nhỏ nhất" liên tiếp tương đương **bỏ lần lượt các phần tử từ đầu mảng đã sắp xếp**, rất tiện để suy luận.

In [3]:
a = [2, 11, 10, 1, 3]
s = sorted(a)
print(s)                # [1, 2, 3, 10, 11]
print(a)                # [2, 11, 10, 1, 3]  (a không đổi)

[1, 2, 3, 10, 11]
[2, 11, 10, 1, 3]


### 3.3. `bisect.bisect_left` trên mảng đã sắp xếp

`bisect.bisect_left(s, k)` trả về **vị trí đầu tiên** mà `k` có thể chèn vào mảng đã sắp xếp `s` để vẫn sắp xếp. Chính là **số phần tử nhỏ hơn `k`** trong `s` (tìm bằng tìm kiếm nhị phân `O(log n)`). `bisect_right` thì đếm số phần tử `<= k`.

In [4]:
s = [1, 2, 3, 10, 11]
print(bisect.bisect_left(s, 10))     # 3: có 3 phần tử < 10 (1, 2, 3)
print(bisect.bisect_right(s, 10))    # 4: có 4 phần tử <= 10
print(bisect.bisect_left(s, 1))      # 0: không có phần tử < 1

3
4
0


### 3.4. `heapq`: lấy phần tử nhỏ nhất nhanh

`heapq` biến list thành **min-heap**: `heapq.heapify(a)` (`O(n)`), `heapq.heappop(a)` lấy ra phần tử nhỏ nhất (`O(log n)`), và `a[0]` luôn là phần tử nhỏ nhất. Rất hợp với thao tác "lấy nhỏ nhất liên tục" của đề.

In [5]:
h = [2, 11, 10, 1, 3]
heapq.heapify(h)
print("nhỏ nhất:", h[0])            # 1
print(heapq.heappop(h), heapq.heappop(h), heapq.heappop(h))   # 1 2 3
print("còn lại (đã thành heap):", sorted(h))                   # [10, 11]

nhỏ nhất: 1
1 2 3
còn lại (đã thành heap): [10, 11]


### 3.5. Đếm bằng `sum` trên biểu thức boolean

`True` tính như `1`, `False` như `0`. Do đó `sum(x < k for x in nums)` đếm số phần tử `x` thỏa `x < k` chỉ trong một dòng.

In [6]:
nums, k = [2, 11, 10, 1, 3], 10
print([x < k for x in nums])         # [True, False, False, True, True]
print(sum(x < k for x in nums))      # 3

[True, False, False, True, True]
3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `nums = [2, 11, 10, 1, 3]`, `k = 10`. Ta **chơi đúng luật** từng bước:

| bước | mảng | nhỏ nhất | nhỏ nhất `>= k`? | hành động |
|---|---|---|---|---|
| 0 | `[2, 11, 10, 1, 3]` | 1 | không | xóa 1 (thao tác 1) |
| 1 | `[2, 11, 10, 3]` | 2 | không | xóa 2 (thao tác 2) |
| 2 | `[11, 10, 3]` | 3 | không | xóa 3 (thao tác 3) |
| 3 | `[11, 10]` | 10 | có | dừng |

Số thao tác = `3`. Nhìn vào: các phần tử bị xóa là `1, 2, 3` — **đúng ba phần tử nhỏ hơn `10`**. Còn `11` và `10` (đều `>= k`) không bao giờ bị xóa. Giả thuyết: **đáp án = số phần tử `< k`**.

In [7]:
nums, k = [2, 11, 10, 1, 3], 10
ops = 0
while min(nums) < k:
    smallest = min(nums)
    nums.remove(smallest)
    ops += 1
    print(f"thao tác {ops}: xóa {smallest}, còn {nums}")
print("tổng:", ops, "| số phần tử < k ban đầu:", sum(x < k for x in [2, 11, 10, 1, 3]))

thao tác 1: xóa 1, còn [2, 11, 10, 3]
thao tác 2: xóa 2, còn [11, 10, 3]
thao tác 3: xóa 3, còn [11, 10]
tổng: 3 | số phần tử < k ban đầu: 3


## 5. Cách 1 — Brute Force / baseline: mô phỏng đúng đề bài

**Ý tưởng.** Lặp: nếu phần tử nhỏ nhất của mảng đã `>= k` thì dừng; còn không, xóa phần tử nhỏ nhất và đếm thêm một thao tác. Ta làm trên **bản sao** để không phá `nums` gốc.

Đề đảm bảo có phần tử `>= k`, nên mảng **không bao giờ cạn** trước khi vòng lặp dừng (nếu mảng rỗng thì `min` sẽ báo lỗi).

In [8]:
def min_operations_simulate(nums: List[int], k: int) -> int:
    arr = list(nums)                  # sao chép để không sửa input
    ops = 0
    while min(arr) < k:               # phần tử nhỏ nhất vẫn chưa đạt
        arr.remove(min(arr))          # xóa một phần tử nhỏ nhất
        ops += 1
    return ops


for nums, k, expected in examples:
    print(min_operations_simulate(nums, k), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
0 (kỳ vọng 0 )
4 (kỳ vọng 4 )


**Độ phức tạp (Cách 1).** Mỗi vòng lặp gọi `min` (`O(n)`) và `remove` (`O(n)`); tối đa `n` vòng ⇒ **Time `O(n²)`**, **Space `O(n)`** (bản sao). Với `n <= 50` thì vô tư.

**Nút thắt:** ta lặp tìm min nhiều lần trong khi kết quả chỉ phụ thuộc vào việc **phần tử nào `< k`**.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát chính.** Thao tác luôn xóa phần tử **nhỏ nhất**. Ta dừng khi phần tử nhỏ nhất `>= k`.

- Chừng nào còn phần tử `< k`, phần tử nhỏ nhất chắc chắn `< k` (vì nhỏ nhất ≤ mọi phần tử), nên ta **phải** xóa tiếp và mỗi lần xóa đi đúng một phần tử `< k`.
- Khi mọi phần tử `< k` đã bị xóa thì phần tử nhỏ nhất còn lại `>= k` ⇒ dừng.
- Phần tử `>= k` **không bao giờ bị xóa**: trước khi chạm tới chúng, ta phải xóa hết phần tử `< k` (chúng nhỏ hơn).

Vậy số thao tác = **số phần tử `< k`**. Không cần mô phỏng!

In [9]:
for nums, k, expected in examples:
    less = [x for x in nums if x < k]
    print(f"nums={nums}, k={k}: các phần tử < k = {less} -> {len(less)} (kỳ vọng {expected})")
    assert len(less) == expected

nums=[2, 11, 10, 1, 3], k=10: các phần tử < k = [2, 1, 3] -> 3 (kỳ vọng 3)
nums=[1, 1, 2, 4, 9], k=1: các phần tử < k = [] -> 0 (kỳ vọng 0)
nums=[1, 1, 2, 4, 9], k=9: các phần tử < k = [1, 1, 2, 4] -> 4 (kỳ vọng 4)


## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp rồi bỏ lần lượt từ đầu

**Giải thích.** Sắp xếp `nums` tăng dần. Xóa phần tử nhỏ nhất = bỏ phần tử ở **đầu**. Dùng chỉ số `i` chạy từ `0`: chừng nào `s[i] < k` thì `i += 1` (tức đã xóa thêm một phần tử). Khi `s[i] >= k` dừng; `i` chính là số thao tác. (Đề đảm bảo có phần tử `>= k` nên `i` không vượt khỏi mảng.)

In [10]:
def min_operations_sorted(nums: List[int], k: int) -> int:
    s = sorted(nums)
    i = 0
    while s[i] < k:               # s[i] là phần tử nhỏ nhất còn lại
        i += 1                    # "xóa" s[i] bằng cách bỏ qua nó
    return i


for nums, k, expected in examples:
    print(min_operations_sorted(nums, k), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
0 (kỳ vọng 0 )
4 (kỳ vọng 4 )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionSorted:
    def minOperations(self, nums: List[int], k: int) -> int:
        nums = sorted(nums)
        i = 0
        while nums[i] < k:
            i += 1
        return i


print(SolutionSorted().minOperations([2, 11, 10, 1, 3], 10))     # 3

3


**Độ phức tạp Cách 2:** **Time `O(n log n)`** (sắp xếp), **Space `O(n)`** (bản sắp xếp).

### Cách 3 — `bisect_left` trên mảng đã sắp xếp

**Giải thích.** Sau khi sắp xếp, các phần tử `< k` tạo thành **một đoạn đầu liên tục**. `bisect_left(s, k)` cho đúng độ dài đoạn đó bằng tìm kiếm nhị phân.

In [12]:
def min_operations_bisect(nums: List[int], k: int) -> int:
    s = sorted(nums)
    return bisect.bisect_left(s, k)


for nums, k, expected in examples:
    print(min_operations_bisect(nums, k), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
0 (kỳ vọng 0 )
4 (kỳ vọng 4 )


**Độ phức tạp Cách 3:** **Time `O(n log n)`** (sắp xếp chiếm phần lớn; bisect chỉ `O(log n)`), **Space `O(n)`**.

### Cách 4 — Min-heap mô phỏng

**Giải thích.** Dùng `heapq` để lấy min nhanh: `heapify` `O(n)`, rồi `while heap[0] < k: heappop`. Đây là bản "mô phỏng nhanh" của Cách 1, hữu ích khi `n` rất lớn và bạn chưa nhìn ra công thức đếm.

In [13]:
def min_operations_heap(nums: List[int], k: int) -> int:
    heap = list(nums)
    heapq.heapify(heap)
    ops = 0
    while heap[0] < k:
        heapq.heappop(heap)
        ops += 1
    return ops


for nums, k, expected in examples:
    print(min_operations_heap(nums, k), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
0 (kỳ vọng 0 )
4 (kỳ vọng 4 )


**Độ phức tạp Cách 4:** **Time `O(n + m log n)`** với `m` là số thao tác (`m <= n`, nên `O(n log n)` trong tệ nhất), **Space `O(n)`**.

### Cách 5 — Đếm trực tiếp (tối ưu và ngắn nhất)

**Giải thích.** Dùng quan sát ở §6: đáp án là số phần tử `< k`. Một lượt duyệt, không sắp xếp, không mảng phụ.

In [14]:
def min_operations_count(nums: List[int], k: int) -> int:
    count = 0
    for x in nums:
        if x < k:
            count += 1
    return count


for nums, k, expected in examples:
    print(min_operations_count(nums, k), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
0 (kỳ vọng 0 )
4 (kỳ vọng 4 )


In [15]:
# Code nộp LeetCode cho Cách 5
class SolutionCount:
    def minOperations(self, nums: List[int], k: int) -> int:
        return sum(x < k for x in nums)


print(SolutionCount().minOperations([1, 1, 2, 4, 9], 9))     # 4

4


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(1)`**. Tối ưu: phải nhìn mọi phần tử ít nhất một lần.

## 8. TRACE từng bước

Ta trace Cách 2 (sắp xếp, con trỏ `i`) và Cách 4 (heap) trên `nums = [2, 11, 10, 1, 3]`, `k = 10`.

In [16]:
def trace_sorted(nums: List[int], k: int) -> int:
    s = sorted(nums)
    print("sau khi sắp xếp:", s, "| k =", k)
    i = 0
    while s[i] < k:
        print(f"i={i}: s[{i}]={s[i]} < {k} -> xóa, thao tác thứ {i + 1}")
        i += 1
    print(f"i={i}: s[{i}]={s[i]} >= {k} -> dừng. Đáp án = {i}")
    return i


assert trace_sorted([2, 11, 10, 1, 3], 10) == 3

sau khi sắp xếp: [1, 2, 3, 10, 11] | k = 10
i=0: s[0]=1 < 10 -> xóa, thao tác thứ 1
i=1: s[1]=2 < 10 -> xóa, thao tác thứ 2
i=2: s[2]=3 < 10 -> xóa, thao tác thứ 3
i=3: s[3]=10 >= 10 -> dừng. Đáp án = 3


In [17]:
def trace_heap(nums: List[int], k: int) -> int:
    heap = list(nums)
    heapq.heapify(heap)
    ops = 0
    print("k =", k)
    while heap[0] < k:
        smallest = heapq.heappop(heap)
        ops += 1
        print(f"pop {smallest} (< {k}) -> ops={ops}, nhỏ nhất mới = {heap[0]}")
    print("nhỏ nhất", heap[0], ">= k -> dừng. Đáp án =", ops)
    return ops


assert trace_heap([2, 11, 10, 1, 3], 10) == 3

k = 10
pop 1 (< 10) -> ops=1, nhỏ nhất mới = 2
pop 2 (< 10) -> ops=2, nhỏ nhất mới = 3
pop 3 (< 10) -> ops=3, nhỏ nhất mới = 10
nhỏ nhất 10 >= k -> dừng. Đáp án = 3


Bảng trace Cách 2 với `[2, 11, 10, 1, 3]`, `k = 10` (mảng sắp xếp `[1, 2, 3, 10, 11]`):

| i | s[i] | s[i] < 10? | hành động | số thao tác sau bước |
|---|---|---|---|---|
| 0 | 1 | có | xóa | 1 |
| 1 | 2 | có | xóa | 2 |
| 2 | 3 | có | xóa | 3 |
| 3 | 10 | không | dừng | 3 |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 5 (đếm trực tiếp): `O(n)` thời gian, `O(1)` bộ nhớ.

In [18]:
from typing import List

class Solution:
    def minOperations(self, nums: List[int], k: int) -> int:
        # mỗi thao tác xóa đúng một phần tử < k, và chỉ phần tử < k mới bị xóa
        return sum(x < k for x in nums)


sol = Solution()
for nums, k, expected in examples:
    print(nums, k, "->", sol.minOperations(nums, k))

[2, 11, 10, 1, 3] 10 -> 3
[1, 1, 2, 4, 9] 1 -> 0
[1, 1, 2, 4, 9] 9 -> 4


## 10. Vì sao thuật toán đúng

- **Mỗi thao tác xóa đúng một phần tử `< k` (khi chưa dừng):** chưa dừng nghĩa là phần tử nhỏ nhất `< k`, và đó chính là phần tử bị xóa.
- **Không bao giờ xóa phần tử `>= k`:** muốn xóa nó thì nó phải là nhỏ nhất, nhưng khi nó là nhỏ nhất thì mọi phần tử còn lại đều `>= nó >= k` ⇒ đã dừng từ trước.
- **Dừng đúng lúc:** khi hết phần tử `< k`. Số lần xóa = số phần tử `< k` ban đầu.
- **Tính tối thiểu:** mỗi phần tử `< k` buộc phải bị xóa (nếu còn lại thì mảng chưa đạt), mỗi thao tác xóa tối đa một phần tử, nên cần **ít nhất** số đó; và đúng số đó là đủ.
- **Vai trò của "có ít nhất một phần tử `>= k`":** đảm bảo mảng không bị xóa sạch (kết quả vẫn có nghĩa).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng `min` + `remove` | `O(n²)` | `O(n)` |
| 2. Sắp xếp + con trỏ | `O(n log n)` | `O(n)` |
| 3. Sắp xếp + `bisect_left` | `O(n log n)` | `O(n)` |
| 4. Min-heap | `O(n log n)` | `O(n)` |
| 5. Đếm trực tiếp | `O(n)` | `O(1)` |

## 12. Bộ test

Ta gom các cách vào `impls` và kiểm tra thêm: các hàm **không được làm thay đổi `nums`** gốc.

In [19]:
sol = Solution()
impls = {
    "simulate": min_operations_simulate,
    "sorted": min_operations_sorted,
    "bisect": min_operations_bisect,
    "heap": min_operations_heap,
    "count": min_operations_count,
    "leetcode": sol.minOperations,
}


def check_all(nums, k, expected, label):
    for key, fn in impls.items():
        data = list(nums)
        actual = fn(data, k)
        assert actual == expected, (label, key, nums, k, actual, expected)
        assert data == nums, ("hàm làm thay đổi input", key)
    print("PASS", label)


for idx, (nums, k, expected) in enumerate(examples, 1):
    check_all(nums, k, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [20]:
check_all([5], 5, 0, "một phần tử đúng bằng k")
check_all([5], 1, 0, "một phần tử lớn hơn k")
check_all([1, 5], 5, 1, "xóa một phần tử")
check_all([3, 3, 3, 3], 3, 0, "mọi phần tử bằng k (không bị xóa)")
check_all([3, 3, 3, 4], 4, 3, "ba phần tử bằng nhau đều < k")
check_all([10**9, 1, 1], 10**9, 2, "giá trị lớn nhất theo đề")
check_all([1, 2, 3, 4, 5], 5, 4, "k bằng phần tử lớn nhất")
check_all([5, 4, 3, 2, 1], 3, 2, "mảng giảm dần")
check_all(list(range(1, 51)), 50, 49, "n = 50, chỉ giữ lại phần tử 50")
check_all([7] * 50, 7, 0, "n = 50, toàn bằng k")

PASS một phần tử đúng bằng k
PASS một phần tử lớn hơn k
PASS xóa một phần tử
PASS mọi phần tử bằng k (không bị xóa)
PASS ba phần tử bằng nhau đều < k
PASS giá trị lớn nhất theo đề
PASS k bằng phần tử lớn nhất
PASS mảng giảm dần
PASS n = 50, chỉ giữ lại phần tử 50
PASS n = 50, toàn bằng k


### 12.2. Phản ví dụ cho các cách làm sai

Hai "ý tưởng gần đúng": (a) đếm phần tử **`<= k`** (nhầm biên), (b) đếm phần tử **`>= k`** rồi lấy `n` trừ (đúng) nhưng sai nếu quên `n`. Ta tạo hàm sai rồi tìm input làm lộ lỗi.

In [21]:
def wrong_leq(nums, k):
    return sum(x <= k for x in nums)


def wrong_count_good(nums, k):
    return sum(x >= k for x in nums)          # quên lấy len(nums) trừ đi


bad1 = ([2, 11, 10, 1, 3], 10)     # có phần tử bằng k = 10
bad2 = ([1, 1, 2, 4, 9], 9)
print(wrong_leq(*bad1), "!=", min_operations_count(*bad1))
print(wrong_count_good(*bad2), "!=", min_operations_count(*bad2))
assert wrong_leq(*bad1) != min_operations_count(*bad1)
assert wrong_count_good(*bad2) != min_operations_count(*bad2)

4 != 3
1 != 4


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Ta sinh input **đúng ràng buộc đề** (có ít nhất một phần tử `>= k`), và so các cách với nhau; Cách 1 (mô phỏng đúng đề) làm chuẩn.

In [22]:
rng = random.Random(3065)
for trial in range(2000):
    n = rng.choice([1, 2, 3, 5, 10, 50])
    hi = rng.choice([3, 10, 100, 10**9])
    nums = [rng.randint(1, hi) for _ in range(n)]
    k = rng.randint(1, max(nums))                 # k <= max(nums) -> luôn có phần tử >= k
    assert 1 <= len(nums) <= 50 and 1 <= k <= 10**9 and any(x >= k for x in nums)
    ref = min_operations_simulate(nums, k)
    for key, fn in impls.items():
        assert fn(list(nums), k) == ref, (key, nums, k)
print("PASS 2000 input ngẫu nhiên: mọi cách khớp mô phỏng đúng đề")

PASS 2000 input ngẫu nhiên: mọi cách khớp mô phỏng đúng đề


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đếm `<= k` thay vì `< k`

In [23]:
def sai_nho_hon_hoac_bang(nums, k):
    return sum(x <= k for x in nums)       # SAI: phần tử bằng k vẫn đạt, không phải xóa


nums, k = [2, 11, 10, 1, 3], 10
print("sai:", sai_nho_hon_hoac_bang(nums, k), "| đúng:", Solution().minOperations(nums, k))

sai: 4 | đúng: 3


**Vì sao sai:** đề yêu cầu "lớn hơn **hoặc bằng** `k`". Phần tử bằng `k` đã hợp lệ nên **không** phải xóa. **Cách sửa:** đếm `x < k`.

### Lỗi 2: Đếm phần tử đạt (`>= k`) thay vì phần tử cần xóa

In [24]:
def sai_dem_phan_tu_dat(nums, k):
    return sum(x >= k for x in nums)       # SAI: đây là số phần tử giữ lại


nums, k = [1, 1, 2, 4, 9], 9
print("sai:", sai_dem_phan_tu_dat(nums, k), "| đúng:", Solution().minOperations(nums, k))

sai: 1 | đúng: 4


**Vì sao sai:** câu hỏi là **số phần tử phải xóa**, tức số phần tử **chưa đạt** (`< k`). **Cách sửa:** `len(nums) - sum(x >= k for x in nums)` hoặc đếm trực tiếp `x < k`.

### Lỗi 3: Vừa duyệt vừa xóa phần tử trong cùng một list

In [25]:
def sai_xoa_khi_duyet(nums, k):
    arr = list(nums)
    ops = 0
    for x in arr:
        if x < k:
            arr.remove(x)          # SAI: sửa list khi đang duyệt -> bỏ sót phần tử
            ops += 1
    return ops


nums, k = [1, 1, 1, 5], 5
print("sai:", sai_xoa_khi_duyet(nums, k), "| đúng:", Solution().minOperations(nums, k))

sai: 2 | đúng: 3


**Vì sao sai:** khi `remove` một phần tử, các phần tử sau dịch lên một chỗ, nhưng vòng `for` vẫn tăng chỉ số ⇒ **bỏ sót phần tử kề sau**. Ở trên chỉ xóa được 2 trong 3 số `1`. **Cách sửa:** đừng sửa list đang duyệt; đếm bằng `sum(...)`, hoặc duyệt `while` với chỉ số được điều khiển cẩn thận.

### Lỗi 4: Xóa "phần tử nhỏ nhất" bằng chỉ số 0 trên mảng chưa sắp xếp

In [26]:
def sai_xoa_dau_mang(nums, k):
    arr = list(nums)
    ops = 0
    while arr[0] < k:              # SAI: arr[0] không phải phần tử nhỏ nhất khi chưa sắp xếp
        arr.pop(0)
        ops += 1
    return ops


nums, k = [10, 1, 5], 5
print("sai:", sai_xoa_dau_mang(nums, k), "| đúng:", Solution().minOperations(nums, k))

sai: 0 | đúng: 1


**Vì sao sai:** thao tác của đề xóa phần tử **nhỏ nhất**, không phải phần tử đầu tiên. Với `[10, 1, 5]`, `arr[0] = 10 >= 5` nên code sai dừng ngay với `0`, trong khi phải xóa phần tử `1`. **Cách sửa:** sắp xếp trước (`sorted`) hoặc đếm trực tiếp `x < k`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "mỗi lần **xóa phần tử nhỏ nhất** cho tới khi mọi phần tử `>= k`" ⇒ đáp án là **số phần tử `< k`** (không cần mô phỏng).
- Khi nghi ngờ công thức: **viết mô phỏng đúng đề (brute force)** rồi so với công thức bằng test ngẫu nhiên. Cách này tìm ra quy luật rất nhanh.
- Thao tác "lấy min liên tục" ⇒ nghĩ ngay **min-heap** hoặc **sắp xếp**.
- Mảng đã sắp xếp + câu hỏi "bao nhiêu phần tử `< k`" ⇒ `bisect_left`.
- Mẹo dạng bài này: chú ý **biên** (`<` hay `<=`): thử input có phần tử bằng đúng `k`.

## 15. Biến thể của bài

1. **Trả về các phần tử còn lại** sau khi xóa (mảng đã lọc).
2. **Mỗi thao tác xóa hai phần tử nhỏ nhất rồi thêm một phần tử mới** — đây là bài **Minimum Operations to Exceed Threshold Value II (3066)**, cần heap thật sự.
3. **Xóa phần tử lớn nhất** thay vì nhỏ nhất, cho tới khi mọi phần tử `<= k`.
4. **Tổng giá trị các phần tử bị xóa**.
5. **Kth Largest Element in an Array (215)**: dùng heap để lấy phần tử nhỏ/lớn nhất liên tục.

In [27]:
def remaining_after_ops(nums, k):
    return sorted(x for x in nums if x >= k)


def ops_remove_largest(nums, k):
    """Xóa phần tử LỚN nhất cho tới khi mọi phần tử <= k: số phần tử > k."""
    return sum(x > k for x in nums)


def removed_sum(nums, k):
    return sum(x for x in nums if x < k)


print(remaining_after_ops([2, 11, 10, 1, 3], 10))     # [10, 11]
print(ops_remove_largest([2, 11, 10, 1, 3], 10))      # 1  (chỉ 11 > 10)
print(removed_sum([2, 11, 10, 1, 3], 10))             # 6  (1 + 2 + 3)
assert remaining_after_ops([2, 11, 10, 1, 3], 10) == [10, 11]
assert ops_remove_largest([2, 11, 10, 1, 3], 10) == 1
assert removed_sum([2, 11, 10, 1, 3], 10) == 6

[10, 11]
1
6


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | `min` + `remove` lặp | `O(n²)` | `O(n)` | hiểu đề, làm chuẩn kiểm tra |
| 2. Sắp xếp + con trỏ | bỏ dần từ đầu mảng | `O(n log n)` | `O(n)` | khi thích tư duy "xóa từ đầu" |
| 3. `bisect_left` | tìm vị trí `k` | `O(n log n)` | `O(n)` | mảng đã sắp xếp sẵn |
| 4. Min-heap | `heappop` đến khi min `>= k` | `O(n log n)` | `O(n)` | bài II, heap thật sự |
| 5. Đếm trực tiếp | `sum(x < k)` | `O(n)` | `O(1)` | **nên nộp** |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `min([4, 2, 9])`, `sorted([3, 1, 2])`, `bisect.bisect_left([1, 3, 3, 5], 3)`, `sum(x < 4 for x in [1, 4, 5, 2])`, `Solution().minOperations([5, 1, 2, 7], 5)`.

**Bài 2 (tính tay).** Với `nums = [9, 4, 6, 1, 8, 3]`, `k = 6`: chơi đúng luật từng bước (xóa nhỏ nhất, ghi lại mảng sau mỗi lần), rồi so với số phần tử `< 6`.

**Bài 3 (điền TODO).** Hoàn thành `min_operations_todo(nums, k)` theo Cách 2 (sắp xếp + con trỏ).

In [28]:
def min_operations_todo(nums, k):
    # TODO 1: sắp xếp nums (không sửa nums gốc)
    i = 0
    # TODO 2: chừng nào phần tử ở chỉ số i còn < k thì tăng i
    # TODO 3: trả về i
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `min_operations_heap2(nums, k)` dùng `heapq` (Cách 4). Sau đó giải thích bằng một câu vì sao có thể thay bằng `sum(x < k ...)`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `min_ops_strict(nums, k)` trả về **số thao tác ít nhất để mọi phần tử còn lại đều `> k`** (lớn hơn **hẳn** `k`, không chấp nhận bằng). (b) Viết `total_removed_value(nums, k)` trả tổng giá trị các phần tử bị xóa; kiểm tra đúng bằng mô phỏng.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `2`, `[1, 2, 3]`, `1` (vị trí đầu tiên của `3`), `2` (là `1` và `2`), `2` (là `1` và `2`). Bài 3: `s = sorted(nums)`, `while s[i] < k: i += 1`.
- **Mức 2 (vừa):** Bài 2: xóa `1` → `[9,4,6,8,3]`; xóa `3` → `[9,4,6,8]`; xóa `4` → `[9,6,8]`; nhỏ nhất `6 >= 6` dừng. Đáp án `3` = số phần tử `< 6` (`1, 3, 4`). Bài 4: `while heap[0] < k: heappop`.
- **Mức 3 (sâu):** Bài 5(a): chỉ cần đổi `<` thành `<=` trong biểu thức đếm: phần tử bằng `k` cũng phải xóa. Đảm bảo vẫn còn phần tử `> k` (tự thêm điều kiện đầu vào). 5(b): tổng các phần tử `< k`, vì đúng những phần tử đó bị xóa.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert min([4, 2, 9]) == 2
assert sorted([3, 1, 2]) == [1, 2, 3]
assert bisect.bisect_left([1, 3, 3, 5], 3) == 1
assert sum(x < 4 for x in [1, 4, 5, 2]) == 2
assert Solution().minOperations([5, 1, 2, 7], 5) == 2

# Bài 2
arr = [9, 4, 6, 1, 8, 3]
steps = []
while min(arr) < 6:
    arr.remove(min(arr))
    steps.append(list(arr))
assert steps == [[9, 4, 6, 8, 3], [9, 4, 6, 8], [9, 6, 8]]
assert len(steps) == sum(x < 6 for x in [9, 4, 6, 1, 8, 3]) == 3


# Bài 3
def min_operations_todo_ans(nums, k):
    s = sorted(nums)
    i = 0
    while s[i] < k:
        i += 1
    return i


for nums, k, expected in examples:
    assert min_operations_todo_ans(nums, k) == expected


# Bài 4
def min_operations_heap2(nums, k):
    heap = list(nums)
    heapq.heapify(heap)
    ops = 0
    while heap[0] < k:
        heapq.heappop(heap)
        ops += 1
    return ops


for nums, k, expected in examples:
    assert min_operations_heap2(nums, k) == expected


# Bài 5(a): mọi phần tử còn lại phải > k  <=>  xóa mọi phần tử <= k
def min_ops_strict(nums, k):
    return sum(x <= k for x in nums)


assert min_ops_strict([2, 11, 10, 1, 3], 10) == 4      # xóa 1, 2, 3, 10; còn [11]
assert min_ops_strict([1, 1, 2, 4, 9], 1) == 2


# Bài 5(b)
def total_removed_value(nums, k):
    return sum(x for x in nums if x < k)


def total_removed_by_simulation(nums, k):
    arr = list(nums)
    total = 0
    while min(arr) < k:
        m = min(arr)
        arr.remove(m)
        total += m
    return total


rng = random.Random(1)
for _ in range(300):
    nums = [rng.randint(1, 20) for _ in range(rng.randint(1, 15))]
    k = rng.randint(1, max(nums))
    assert total_removed_value(nums, k) == total_removed_by_simulation(nums, k)
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (xóa phần tử nhỏ nhất đến khi mọi phần tử `>= k` ⇒ đếm phần tử `< k`)

```python
from typing import List

class Solution:
    def minOperations(self, nums: List[int], k: int) -> int:
        return sum(x < k for x in nums)
```

**Checklist**

- [ ] Đếm phần tử **`< k`** (không phải `<= k`, không phải `>= k`)?
- [ ] Hiểu vì sao chỉ phần tử `< k` bị xóa và không bao giờ xóa phần tử `>= k`?
- [ ] Không sửa list trong lúc duyệt `for`?
- [ ] Nếu dùng mô phỏng: lấy **nhỏ nhất** thật sự (`min` / `sorted` / heap), không phải phần tử đầu?
- [ ] Đã thử `k` bằng phần tử nhỏ nhất, bằng phần tử lớn nhất, và có phần tử đúng bằng `k`?
- [ ] Skeleton đúng `class Solution` / `def minOperations(self, nums: List[int], k: int) -> int`?